In [1]:
import pandas as pd
import numpy as np
import os
import secrets
import openmatrix as omx

## File cropping for testing
The idea here is for us to be able to quickly iterate through the model using a small subset of the data for easier development and testing.  Set the files and zones below and run the notebook.

In [2]:
# read full input data
input_folder = r".\metro_data"
output_folder = r".\metro_data_cropped"
full_households = pd.read_csv(os.path.join(input_folder, 'households.csv'))
full_persons = pd.read_csv(os.path.join(input_folder, 'persons.csv'))
full_landuse = pd.read_csv(os.path.join(input_folder, 'land_use.csv'))

input_skim_folder = r".\metro_data\skims"
skim_list = [
    "autoSkims__EA.omx",
    "autoSkims__AM.omx",
    "autoSkims__MD.omx",
    "autoSkims__PM.omx",
    "autoSkims__EV.omx",
    "transitSkims__EA.omx",
    "transitSkims__AM.omx",
    "transitSkims__MD.omx",
    "transitSkims__PM.omx",
    "transitSkims__EV.omx",
    "walkSkim.omx"
]
maz_skim_file = "maz_maz_walk.csv"

# optional since this can take a while (actually only about 1 minute!)
CROP_SKIMS = True
# downtown and a little bit of the northwest + some externals in NE Vancouver
tazs_to_crop = list(range(1, 51)) + list(range(2000, 2010))

In [3]:
cropped_landuse = full_landuse[full_landuse['TAZ'].isin(tazs_to_crop)].copy()
cropped_households = full_households[full_households['TAZ'].isin(tazs_to_crop)].copy()
cropped_persons = full_persons[full_persons['household_id'].isin(cropped_households['household_id'])].copy()

In [4]:
# obfuscate the employment data
emp_colummns = [col for col in cropped_landuse.columns if col.startswith('EMP')]
for col in emp_colummns:
    # scale employment by +/- 0-50% 
    cropped_landuse[col] = cropped_landuse[col].apply(
        lambda x: x * (0.5 + secrets.randbelow(10000) / 10000)
    ).fillna(secrets.randbelow(100)).astype(int)  

In [5]:
# check there is some enrollment and external counts in the cropped dataset
for col in ['ENROLL_K8','ENROLL_912','ENROLL_COLL', "external_work", "external_nonwork"]:
    if cropped_landuse[col].sum() == 0:
        raise ValueError(f"Column {col} has no data in the cropped land use dataset.")

In [6]:
print(f"Cropped land use dataset has {len(cropped_landuse)} MAZs")
print(f"Cropped households dataset has {len(cropped_households)} households")
print(f"Cropped persons dataset has {len(cropped_persons)} persons")
# save the cropped datasets
cropped_landuse.to_csv(os.path.join(output_folder, 'land_use.csv'), index=False)
cropped_households.to_csv(os.path.join(output_folder, 'households.csv'), index=False)
cropped_persons.to_csv(os.path.join(output_folder, 'persons.csv'), index=False)

Cropped land use dataset has 712 MAZs
Cropped households dataset has 35986 households
Cropped persons dataset has 59632 persons


In [7]:
if CROP_SKIMS:
    # crop omx skim files
    taz_labels = cropped_landuse.sort_values("TAZ")['TAZ'].unique() # TAZ zone_ids in omx index order
    tazs_indexes = taz_labels - 1  # index of TAZ in skim (zero-based, no mapping)

    for omx_infile_name in skim_list:
        skim_data_type = np.float32

        omx_in = omx.open_file(os.path.join(input_skim_folder, omx_infile_name))
        omx_out = omx.open_file(os.path.join(output_folder, omx_infile_name), "w")
        print(f"omx_in shape {omx_in.shape()}")

        omx_out.create_mapping("ZONE", taz_labels)

        for mat_name in omx_in.list_matrices():

            # make sure we have a vanilla numpy array, not a CArray
            m = np.asanyarray(omx_in[mat_name]).astype(skim_data_type)
            m = m[tazs_indexes, :][:, tazs_indexes]
            print(f"{mat_name} {m.shape}")

            omx_out[mat_name] = m

        omx_in.close()
        omx_out.close()

omx_in shape (2162, 2162)
SOV_H_COST__EA (60, 60)
SOV_H_DIST__EA (60, 60)
SOV_H_TIME__EA (60, 60)
SOV_L_COST__EA (60, 60)
SOV_L_DIST__EA (60, 60)
SOV_L_TIME__EA (60, 60)
SOV_M_COST__EA (60, 60)
SOV_M_DIST__EA (60, 60)
SOV_M_TIME__EA (60, 60)
SR2_H_COST__EA (60, 60)
SR2_H_DIST__EA (60, 60)
SR2_H_TIME__EA (60, 60)
SR2_L_COST__EA (60, 60)
SR2_L_DIST__EA (60, 60)
SR2_L_TIME__EA (60, 60)
SR2_M_COST__EA (60, 60)
SR2_M_DIST__EA (60, 60)
SR2_M_TIME__EA (60, 60)
SR3_H_COST__EA (60, 60)
SR3_H_DIST__EA (60, 60)
SR3_H_TIME__EA (60, 60)
SR3_L_COST__EA (60, 60)
SR3_L_DIST__EA (60, 60)
SR3_L_TIME__EA (60, 60)
SR3_M_COST__EA (60, 60)
SR3_M_DIST__EA (60, 60)
SR3_M_TIME__EA (60, 60)
omx_in shape (2162, 2162)
SOV_H_COST__AM (60, 60)
SOV_H_DIST__AM (60, 60)
SOV_H_TIME__AM (60, 60)
SOV_L_COST__AM (60, 60)
SOV_L_DIST__AM (60, 60)
SOV_L_TIME__AM (60, 60)
SOV_M_COST__AM (60, 60)
SOV_M_DIST__AM (60, 60)
SOV_M_TIME__AM (60, 60)
SR2_H_COST__AM (60, 60)
SR2_H_DIST__AM (60, 60)
SR2_H_TIME__AM (60, 60)
SR2_L_COST__

In [8]:
if CROP_SKIMS:
    # crop the maz-maz walk skim
    maz_maz_walk = pd.read_csv(os.path.join(input_skim_folder, maz_skim_file))
    cropped_maz_maz_walk = maz_maz_walk[maz_maz_walk['OMAZ'].isin(cropped_landuse.MAZ) & maz_maz_walk['DMAZ'].isin(cropped_landuse.MAZ)]
    # FIXME this should be renamed in the maz-maz walk creation script
    maz_maz_walk.rename(columns={'DISTWALK': 'WLK_DIST'}, inplace=True)
    cropped_maz_maz_walk.to_csv(os.path.join(output_folder, maz_skim_file), index=False)

In [40]:
skm = omx.open_file(os.path.join(output_folder, 'transitSkims__AM.omx'), "r")
print(skm.list_matrices())
skm.close()

['KTW_ACC__AM', 'KTW_AST__AM', 'KTW_AUX__AM', 'KTW_BRT__AM', 'KTW_BUS__AM', 'KTW_CRT__AM', 'KTW_EGR__AM', 'KTW_FWT__AM', 'KTW_LRT__AM', 'KTW_RST__AM', 'KTW_SCR__AM', 'KTW_TIV__AM', 'KTW_VTC__AM', 'KTW_XFR__AM', 'KTW_XWT__AM', 'WTK_ACC__AM', 'WTK_AST__AM', 'WTK_AUX__AM', 'WTK_BRT__AM', 'WTK_BUS__AM', 'WTK_CRT__AM', 'WTK_EGR__AM', 'WTK_FWT__AM', 'WTK_LRT__AM', 'WTK_RST__AM', 'WTK_SCR__AM', 'WTK_TIV__AM', 'WTK_VTC__AM', 'WTK_XFR__AM', 'WTK_XWT__AM', 'WTW_ACC__AM', 'WTW_AST__AM', 'WTW_AUX__AM', 'WTW_BRT__AM', 'WTW_BUS__AM', 'WTW_CRT__AM', 'WTW_EGR__AM', 'WTW_FWT__AM', 'WTW_LRT__AM', 'WTW_RST__AM', 'WTW_SCR__AM', 'WTW_TIV__AM', 'WTW_VTC__AM', 'WTW_XFR__AM', 'WTW_XWT__AM']


In [11]:
tazs[tazs['NO'].isin(tazs_to_crop)].explore()